In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os

In [ ]:
d = 3 #hilbert space dimensions

input_dim = d**2
output_dim = d**2
save_dir = f'/kaggle/working/dataset_d{d}'

X_train =np.load(os.path.join(save_dir, "X_train.npy"))
Y_train =np.load(os.path.join(save_dir, "Y_train.npy"))
X_val=np.load(os.path.join(save_dir, "X_val.npy"))
Y_val=np.load(os.path.join(save_dir, "Y_val.npy"))
X_test=np.load(os.path.join(save_dir, "X_test.npy"))
Y_test=np.load(os.path.join(save_dir, "Y_test.npy"))

In [ ]:
import numpy as np

def corrupt_data(X, gamma, method='random'):
    """
    Corrupt a fraction gamma of the samples in X.
    method: 'random' – replace with a random probability vector from Dirichlet.
            'swap'   – replace with the vector of a randomly chosen other sample.
    """
    n = X.shape[0]
    X_corr = X.copy()
    mask = np.random.rand(n) < gamma
    n_corr = mask.sum()
    if method == 'random':
        # Dirichlet(1,1,...,1) gives uniform random probabilities
        X_corr[mask] = np.random.dirichlet(np.ones(X.shape[1]), size=n_corr)
    elif method == 'swap':
        # For each corrupted sample, pick a random different index and swap
        for idx in np.where(mask)[0]:
            other = np.random.choice([i for i in range(n) if i != idx])
            X_corr[idx] = X[other]
    return X_corr

gamma = 0.1
X_test = corrupt_data(X_test, gamma, method='random')

In [ ]:
d = 3 #hilbert space dimensions

input_dim = d**2
output_dim = d**2

model =  models.Sequential(name=f"QST{d}")

model.add(layers.InputLayer(input_shape = (input_dim,)))

model.add(layers.Dense(200,activation ='relu'))
model.add(layers.Dense(180,activation ='relu'))
model.add(layers.Dense(180,activation ='relu'))
model.add(layers.Dense(160,activation ='relu'))
model.add(layers.Dense(160,activation ='relu'))
model.add(layers.Dense(160,activation ='relu'))
model.add(layers.Dense(160,activation ='relu'))
model.add(layers.Dense(200,activation ='relu'))

model.add(layers.Dense(output_dim, activation = 'tanh'))

In [ ]:
optimizer = tf.keras.optimizers.Nadam(
    learning_rate = 0.001,
    beta_1 = 0.9,
    beta_2 = 0.999,
    epsilon = 1e-7
)

model.compile(optimizer=optimizer,
             loss='mse',
             metrics=['mae'])

model.summary()

In [ ]:
early_stop= callbacks.EarlyStopping(
    monitor='val_loss',
    patience = 200,
    restore_best_weights= True,

    verbose=2
)


checkpoint = callbacks.ModelCheckpoint(
    filepath = f"best_moodel_d{d}.keras",
    monitor = 'vals_loss',
    save_best_only = True,
    verbose = 1
)

history = model.fit(
    X_train, Y_train,
    batch_size = 100,
    epochs = 2000,
    validation_data = (X_val, Y_val),
    callbacks = [early_stop, checkpoint],
    verbose = 1
)

test_loss, test_mae  = model.evaluate(X_test, Y_test, verbose = 0)

print(f"nTest loss (MSE): {test_loss:.6f}")
print(f"Test MAE: {test_mae:.6f}")

model.save(f"final_model_d{d}.keras")

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have 'history' from model.fit()
# history = model.fit(...)

# Plot training & validation loss
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

# Optional: save the figure
plt.savefig('loss_curve.png')

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have the 'history' object from model.fit()
# history = model.fit(...)

# Extract the history dictionary
hist = history.history

# Check available metrics (in case you added more)
print("Metrics recorded:", hist.keys())

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Left plot: Loss (MSE) ---
ax1.plot(hist['loss'], label='Training Loss', linewidth=1.5)
ax1.plot(hist['val_loss'], label='Validation Loss', linewidth=1.5)
ax1.set_title('Loss (Mean Squared Error)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE')
ax1.legend()
ax1.grid(True)

# --- Right plot: MAE ---
ax2.plot(hist['mae'], label='Training MAE', linewidth=1.5)
ax2.plot(hist['val_mae'], label='Validation MAE', linewidth=1.5)
ax2.set_title('Mean Absolute Error')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

**D = 7**